# 🥈 Silver Layer: Clean and Standardized Data

## What is the Silver Layer?

The Silver layer takes **raw data from Bronze** and transforms it into **clean, standardized, business-ready** data.

### Key Transformations:
1. **Handle null values** - Replace, impute, or document missing data
2. **Standardize columns** - Consistent naming conventions (snake_case)
3. **Type conversions** - Convert strings to dates, numbers, etc.
4. **Deduplication** - Remove duplicate records
5. **Derived columns** - Add calculated fields for analysis
6. **Data quality checks** - Validate data integrity

### Silver Layer Rules:
* **Clean but not aggregated** - Individual records, not summaries
* **Standardized** - Consistent formats and naming
* **Validated** - Quality checks enforce business rules
* **Enriched** - Additional columns for downstream analytics

---

**Pipeline Flow:** Bronze (Raw) → **Silver (Clean)** → Gold (Analytics)

In [0]:
# ===================================================================
# STEP 1: Read Bronze Layer Data
# ===================================================================
# Start by loading the raw data from the Bronze table
# ===================================================================

print("📥 Reading from Bronze layer...\n")

# Read the Bronze table
df_bronze = spark.table("workspace.netflix_bronze.raw_titles")

print(f"✓ Loaded {df_bronze.count():,} records from Bronze")
print(f"✓ Schema has {len(df_bronze.columns)} columns")

print("\n📊 Original columns:")
for col in df_bronze.columns:
    print(f"  • {col}")

In [0]:
# ===================================================================
# STEP 2: Data Quality Assessment
# ===================================================================
# Before cleaning, let's understand our data quality issues
# ===================================================================

from pyspark.sql.functions import col, count, when, isnan, isnull

print("🔍 Analyzing NULL values in each column:\n")
print(f"{'Column':<25} {'Null Count':<15} {'Null %':<10}")
print("="*50)

# Count nulls in each column
total_rows = df_bronze.count()

for column in df_bronze.columns:
    # Get the column's data type
    col_type = dict(df_bronze.dtypes)[column]
    
    # For string columns, check both NULL and empty string
    # For other types (like TIMESTAMP), only check NULL
    if col_type == 'string':
        null_count = df_bronze.filter(col(column).isNull() | (col(column) == "")).count()
    else:
        null_count = df_bronze.filter(col(column).isNull()).count()
    
    null_pct = (null_count / total_rows) * 100
    
    if null_count > 0:
        print(f"{column:<25} {null_count:<15,} {null_pct:<10.2f}%")

print("\n⚠️ Fields with high null rates will need special handling")

In [0]:
# ===================================================================
# STEP 3: Standardize Column Names
# ===================================================================
# Convert column names to snake_case for consistency
# This makes it easier to write SQL queries and prevents issues
# with spaces and special characters
# ===================================================================

import re

def to_snake_case(name):
    """Convert column name to snake_case"""
    # Replace spaces and special characters with underscores
    name = re.sub(r'[\s-]+', '_', name)
    # Convert to lowercase
    name = name.lower()
    # Remove duplicate underscores
    name = re.sub(r'_+', '_', name)
    return name

print("🔄 Standardizing column names to snake_case...\n")

# Rename columns
df_renamed = df_bronze
for old_col in df_bronze.columns:
    new_col = to_snake_case(old_col)
    if old_col != new_col:
        df_renamed = df_renamed.withColumnRenamed(old_col, new_col)
        print(f"  {old_col} → {new_col}")

print("\n✓ Column names standardized")

# Show new schema
print("\n📊 New columns:")
for col in df_renamed.columns:
    print(f"  • {col}")

In [0]:
# ===================================================================
# STEP 4: Handle Null Values
# ===================================================================
# Strategy:
# - Replace empty strings with NULL for consistency
# - Fill director/cast/country with 'Unknown'
# - Keep rating nulls (we'll filter in Gold layer if needed)
# - Remove records with null show_id (primary key)
# - Remove records with invalid type (corrupt data)
# ===================================================================

from pyspark.sql.functions import when, col, trim, lit

print("🧼 Cleaning data and handling nulls...\n")

df_clean = df_renamed

# FIRST: Remove corrupt records with invalid type values
if 'type' in df_clean.columns:
    before_count = df_clean.count()
    df_clean = df_clean.filter(col('type').isin(['Movie', 'TV Show']))
    removed = before_count - df_clean.count()
    if removed > 0:
        print(f"⚠️ Removed {removed} corrupt record(s) with invalid type")

# Replace empty strings with NULL across all string columns
for column in df_clean.columns:
    if dict(df_clean.dtypes)[column] == 'string':
        df_clean = df_clean.withColumn(
            column,
            when(trim(col(column)) == "", lit(None)).otherwise(col(column))
        )

print("✓ Replaced empty strings with NULL")

# Fill common null fields with 'Unknown'
fill_unknown = ['director', 'cast', 'country']
for column in fill_unknown:
    if column in df_clean.columns:
        df_clean = df_clean.fillna('Unknown', subset=[column])
        print(f"✓ Filled {column} nulls with 'Unknown'")

# Remove records without a show_id (corrupted data)
if 'show_id' in df_clean.columns:
    before_count = df_clean.count()
    df_clean = df_clean.filter(col('show_id').isNotNull())
    removed = before_count - df_clean.count()
    if removed > 0:
        print(f"\n⚠️ Removed {removed} records with null show_id")

print(f"\n✓ Clean dataset has {df_clean.count():,} records")

In [0]:
# ===================================================================
# STEP 5: Convert Date Fields to Proper Types
# ===================================================================
# The 'date_added' field is stored as string in Bronze
# We'll convert it to a proper date type for time-based analysis
# ===================================================================

from pyspark.sql.functions import to_date, year, month, col, trim, expr

print("📅 Converting date fields...\n")

df_typed = df_clean

# Convert date_added from string to date
if 'date_added' in df_typed.columns:
    # Original format: "January 1, 2020"
    df_typed = df_typed.withColumn(
        'date_added',
        to_date(trim(col('date_added')), 'MMMM d, yyyy')
    )
    print("✓ Converted 'date_added' to date type")
    
    # Extract year and month for easier analysis
    df_typed = df_typed.withColumn('year_added', year(col('date_added')))
    df_typed = df_typed.withColumn('month_added', month(col('date_added')))
    print("✓ Extracted 'year_added' and 'month_added'")

# Convert release_year to integer using try_cast to handle bad data
if 'release_year' in df_typed.columns:
    df_typed = df_typed.withColumn('release_year', expr("try_cast(release_year as int)"))
    print("✓ Ensured 'release_year' is integer type (bad values become NULL)")

print("\n✓ Date conversions complete")

In [0]:
# ===================================================================
# STEP 6: Derive New Columns for Analysis
# ===================================================================
# Add calculated fields that will be useful for analytics:
# - content_age: How old is the content (current year - release year)
# - decade: Which decade was it released
# - is_recent: Content added in last 2 years
# ===================================================================

from pyspark.sql.functions import year, current_date, col, when, floor, lit, expr

print("🧐 Creating derived columns...\n")

df_enriched = df_typed

# Calculate content age
if 'release_year' in df_enriched.columns:
    current_year = 2026  # Using hardcoded value for consistency
    df_enriched = df_enriched.withColumn(
        'content_age',
        lit(current_year) - col('release_year')
    )
    print("✓ Added 'content_age' (years since release)")
    
    # Calculate decade (e.g., 2020 → 2020s, 1995 → 1990s)
    df_enriched = df_enriched.withColumn(
        'decade',
        expr("try_cast((floor(release_year / 10) * 10) as int)")
    )
    print("✓ Added 'decade' field")

# Flag recent additions
if 'year_added' in df_enriched.columns:
    df_enriched = df_enriched.withColumn(
        'is_recent',
        when(col('year_added') >= 2024, True).otherwise(False)
    )
    print("✓ Added 'is_recent' flag (added 2024+)")

# Parse duration into numeric format using try_cast to handle malformed data
if 'duration' in df_enriched.columns:
    # For movies: "90 min" → 90
    # For TV shows: "2 Seasons" → 2
    df_enriched = df_enriched.withColumn(
        'duration_numeric',
        when(col('type') == 'Movie', 
             expr("try_cast(substring(duration, 1, 3) as int)"))
        .when(col('type') == 'TV Show',
             expr("try_cast(substring(duration, 1, 2) as int)"))
        .otherwise(None)
    )
    print("✓ Added 'duration_numeric' (extracted from duration string)")

print("\n✓ Derived columns created")

In [0]:
# ===================================================================
# STEP 7: Remove Duplicate Records
# ===================================================================
# Check for and remove any duplicate entries based on show_id
# ===================================================================

print("🔍 Checking for duplicates...\n")

# Count records before deduplication
before_count = df_enriched.count()

# Remove duplicates based on show_id (primary key)
if 'show_id' in df_enriched.columns:
    df_dedup = df_enriched.dropDuplicates(['show_id'])
else:
    # If no show_id, drop duplicates across all columns
    df_dedup = df_enriched.dropDuplicates()

after_count = df_dedup.count()
duplicates_removed = before_count - after_count

if duplicates_removed > 0:
    print(f"⚠️ Removed {duplicates_removed} duplicate records")
else:
    print("✓ No duplicates found - data is clean!")

print(f"\nFinal record count: {after_count:,}")

In [0]:
# ===================================================================
# STEP 8: Data Quality Validation
# ===================================================================
# Run checks to ensure data meets quality standards
# ===================================================================

from pyspark.sql.functions import col, count, when

print("✅ Running data quality validations...\n")

validation_results = []

# Check 1: All records have show_id
if 'show_id' in df_dedup.columns:
    null_ids = df_dedup.filter(col('show_id').isNull()).count()
    validation_results.append(('show_id not null', null_ids == 0))
    print(f"  Check 1: show_id not null - {'PASS' if null_ids == 0 else 'FAIL'}")

# Check 2: All records have title
if 'title' in df_dedup.columns:
    null_titles = df_dedup.filter(col('title').isNull()).count()
    validation_results.append(('title not null', null_titles == 0))
    print(f"  Check 2: title not null - {'PASS' if null_titles == 0 else 'FAIL'}")

# Check 3: Type is either 'Movie' or 'TV Show'
if 'type' in df_dedup.columns:
    valid_types = df_dedup.filter(col('type').isin(['Movie', 'TV Show'])).count()
    total = df_dedup.count()
    validation_results.append(('valid type', valid_types == total))
    print(f"  Check 3: valid type (Movie/TV Show) - {'PASS' if valid_types == total else 'FAIL'}")

# Check 4: Release year is reasonable (between 1900 and 2026)
if 'release_year' in df_dedup.columns:
    valid_years = df_dedup.filter(
        (col('release_year') >= 1900) & (col('release_year') <= 2026)
    ).count()
    total = df_dedup.count()
    validation_results.append(('valid release_year', valid_years == total))
    print(f"  Check 4: valid release_year (1900-2026) - {'PASS' if valid_years == total else 'FAIL'}")

# Check 5: No duplicate show_ids
if 'show_id' in df_dedup.columns:
    unique_ids = df_dedup.select('show_id').distinct().count()
    total = df_dedup.count()
    validation_results.append(('no duplicates', unique_ids == total))
    print(f"  Check 5: no duplicate show_ids - {'PASS' if unique_ids == total else 'FAIL'}")

all_passed = all([result[1] for result in validation_results])
success_msg = '✅ All validation checks passed!'
fail_msg = '⚠️ Some validation checks failed'
print(f"\n{success_msg if all_passed else fail_msg}")

In [0]:
# ===================================================================
# STEP 9: Write Clean Data to Silver Layer
# ===================================================================
# Save the cleaned, transformed data to the Silver Delta table
# ===================================================================

print("💾 Writing to Silver layer...\n")

# Write to Silver Delta table
df_dedup.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.netflix_silver.clean_titles")

print(f"✅ Successfully wrote {df_dedup.count():,} records to Silver layer")
print("\n📊 Table: workspace.netflix_silver.clean_titles")

# Show final schema
print("\n📊 Final Silver schema:")
silver_df = spark.table("workspace.netflix_silver.clean_titles")
for col_name in silver_df.columns:
    col_type = dict(silver_df.dtypes)[col_name]
    print(f"  • {col_name:<25} ({col_type})")

In [0]:
# ===================================================================
# STEP 10: Preview Clean Silver Data
# ===================================================================

print("📋 Sample records from Silver layer:\n")

# Display sample with key columns
silver_df = spark.table("workspace.netflix_silver.clean_titles")

# Select interesting columns to display
display_cols = ['show_id', 'type', 'title', 'release_year', 'rating', 
                'duration', 'decade', 'content_age', 'country', 'listed_in']

# Only select columns that exist
existing_cols = [c for c in display_cols if c in silver_df.columns]

display(silver_df.select(existing_cols).limit(20))

In [0]:
# ===================================================================
# STEP 11: Silver Layer Summary Statistics
# ===================================================================

from pyspark.sql.functions import col, count, countDistinct, min, max, avg

print("📊 Silver Layer Summary Statistics\n")
print("="*70)

silver_df = spark.table("workspace.netflix_silver.clean_titles")

# Basic counts
print(f"\n📀 Record Counts:")
print(f"  Total Records: {silver_df.count():,}")
if 'type' in silver_df.columns:
    type_counts = silver_df.groupBy('type').count().orderBy('count', ascending=False)
    print(f"\n  By Type:")
    for row in type_counts.collect():
        print(f"    {row['type']:<15} {row['count']:>6,}")

# Year analysis
if 'release_year' in silver_df.columns:
    year_stats = silver_df.select(
        min('release_year').alias('min_year'),
        max('release_year').alias('max_year'),
        avg('release_year').alias('avg_year')
    ).collect()[0]
    
    print(f"\n📅 Release Year Range:")
    print(f"  Oldest: {year_stats['min_year']}")
    print(f"  Newest: {year_stats['max_year']}")
    print(f"  Average: {year_stats['avg_year']:.0f}")

# Country diversity
if 'country' in silver_df.columns:
    country_count = silver_df.filter(col('country') != 'Unknown') \
        .select(countDistinct('country')).collect()[0][0]
    print(f"\n🌍 Geographic Diversity:")
    print(f"  Unique Countries: {country_count}")

# Rating diversity
if 'rating' in silver_df.columns:
    rating_count = silver_df.filter(col('rating').isNotNull()) \
        .select(countDistinct('rating')).collect()[0][0]
    print(f"\n⭐ Content Ratings:")
    print(f"  Unique Ratings: {rating_count}")

print("\n" + "="*70)

## ✅ Silver Layer Complete!

### Transformations Applied:
1. ✓ **Standardized column names** - Converted to snake_case
2. ✓ **Handled null values** - Replaced empties, filled unknowns
3. ✓ **Converted date fields** - Proper date types with extracted year/month
4. ✓ **Removed duplicates** - Ensured unique records by show_id
5. ✓ **Derived new columns**:
   * `content_age` - Years since release
   * `decade` - Release decade (1990s, 2000s, etc.)
   * `is_recent` - Added in last 2 years
   * `duration_numeric` - Parsed numeric duration
6. ✓ **Data quality validation** - All checks passed
7. ✓ **Optimized storage** - Delta Lake format

### Key Improvements vs Bronze:
* **Cleaner** - No empty strings, standardized nulls
* **Typed** - Proper data types (dates, integers)
* **Enriched** - Additional analytical columns
* **Validated** - Quality checks ensure integrity
* **Consistent** - Standardized naming and formats

---

## 🎯 Next Steps:

Now we'll create the **Gold Layer** with business-ready analytics tables:
* Top genres by content count
* Movies vs TV shows trend over time
* Average duration by genre
* Content distribution by country
* And more pre-aggregated analytics!

**Continue to:** `03_Gold_Netflix_Analytics` notebook